<a href="https://colab.research.google.com/github/ArturStachnik/WorkshopPaleoML/blob/main/sesion_2/H2_del_dato_al_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Del dato al modelo

**Machine learning y paleoclima · Red Paleonet · Granada, 2026**

Una función de transferencia de principio a fin: a partir de qué foraminíferos hay en el sedimento, estimar la temperatura del agua en superficie. Es el mismo recorrido de las diapositivas, ahora con datos reales:

| | Bloque | Qué se hace aquí |
|---|---|---|
| 1 | **Datos** | Encontrar y quitar los errores, mirar el conjunto, elegir y preparar los predictores |
| 2 | **Algoritmo** | Elegir un modelo y ver qué pasa si no se separan los datos |
| 3 | **Entrenamiento** | Mover los hiperparámetros y los predictores y ver cómo cambia el error |
| 4 | **Validación** | Comprobar cuánto se puede fiar uno del error que sale |

**No hay que escribir código.** Cada celda tiene un botón de play; las que tienen controles a la derecha (desplegables, deslizadores) se cambian con el ratón y se vuelven a ejecutar. Las que tienen controles debajo de la figura se actualizan solas.

El código está escondido. Para verlo, doble clic en la celda.

---
## 0. Preparar el cuaderno

**Qué hacer:** ejecutar la celda. Tarda unos segundos: carga las librerías, la tipografía del taller y el mapa de costas.

Primero, como en la sesión 1: **Archivo → Guardar una copia en Drive**.

In [ ]:
#@title Preparar el cuaderno
import os, json, time, shutil, tempfile, logging, warnings, textwrap, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.collections import PolyCollection
from matplotlib.path import Path
import ipywidgets as w
from IPython.display import display, Markdown, clear_output
from sklearn.model_selection import train_test_split, KFold, GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None
warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
%config InlineBackend.figure_format = "retina"

REPO = "https://raw.githubusercontent.com/ArturStachnik/WorkshopPaleoML/main/"


def ruta(fichero):
    """El fichero del repositorio: del disco si está descargado, de GitHub si no."""
    for base in ("", "../"):
        if os.path.exists(base + fichero):
            return base + fichero
    return REPO + fichero


def leer_texto(fichero):
    origen = ruta(fichero)
    if origen.startswith("http"):
        return urllib.request.urlopen(origen, timeout=60).read().decode("utf-8")
    return open(origen, encoding="utf-8").read()


# --- tipografía y colores del taller ---
FUENTES = os.path.join(tempfile.gettempdir(), "fuentes_taller")
os.makedirs(FUENTES, exist_ok=True)
for f in ("DMSans.ttf", "DMSans-Italic.ttf", "DMSerifDisplay.ttf"):
    destino = os.path.join(FUENTES, f)
    try:
        if not os.path.exists(destino):
            origen = ruta("recursos/fuentes/" + f)
            if origen.startswith("http"):
                urllib.request.urlretrieve(origen, destino)
            else:
                shutil.copy(origen, destino)
        font_manager.fontManager.addfont(destino)
    except Exception:
        pass

TINTA, GRIS, ARENA, AZUL, ROJO, ORO, VERDE = "#1D2B36", "#8A8F94", "#E8E2D6", "#1F6E8C", "#C2462E", "#D9A441", "#6B8F71"
SST_CMAP = LinearSegmentedColormap.from_list(
    "sst", ["#16405A", AZUL, "#8DB9C9", "#EFE6D6", "#E6A592", ROJO, "#7E2A1A"])
plt.rcParams.update({
    "font.family": ["DM Sans", "DejaVu Sans"], "font.size": 11,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA, "axes.titlesize": 13,
    "axes.titlelocation": "left", "axes.titlecolor": TINTA, "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#EEEAE2", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": GRIS, "ytick.color": GRIS, "xtick.labelcolor": TINTA, "ytick.labelcolor": TINTA,
    "figure.dpi": 100, "figure.facecolor": "white", "axes.facecolor": "white",
    "legend.frameon": False, "text.color": TINTA,
})

# --- el mapa ---
_tierra = json.loads(leer_texto("datos/tierra_ne_110m.geojson"))
POLIGONOS = []
for f in _tierra["features"]:
    g = f["geometry"]
    for p in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
        POLIGONOS.append(np.array(p[0]))
_CAMINOS = [Path(p) for p in POLIGONOS]


def fondo_mapa(ax, lat=(-80, 90)):
    ax.add_collection(PolyCollection(POLIGONOS, facecolor=ARENA, edgecolor="none", zorder=0))
    ax.set_xlim(-180, 180); ax.set_ylim(*lat); ax.set_aspect("equal")
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xticks([-120, -60, 0, 60, 120]); ax.set_yticks([-60, -30, 0, 30, 60])
    ax.tick_params(length=0, labelsize=9, labelcolor=GRIS)


def en_tierra(lon, lat):
    """Dentro de tierra y a más de ~100 km de la costa (el mapa es grueso)."""
    lon, lat = np.asarray(lon, float), np.asarray(lat, float)
    dentro = np.ones(len(lon), bool)
    for dx, dy in ((0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)):
        xy = np.column_stack([lon + dx, lat + dy])
        algo = np.zeros(len(lon), bool)
        for c in _CAMINOS:
            algo |= c.contains_points(xy)
        dentro &= algo
    return dentro


def corto(nombre):
    """Neogloboquadrina_pachyderma -> N. pachyderma"""
    p = nombre.split("_")
    return p[0][0] + ". " + " ".join(p[1:])


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))


def r2(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2))


def obs_pred(ax, y, p, color, titulo):
    ax.scatter(y, p, s=7, color=color, alpha=0.45, lw=0)
    ax.plot([-4, 32], [-4, 32], color=TINTA, lw=1, ls=(0, (4, 3)))
    ax.set_xlim(-4, 32); ax.set_ylim(-4, 32); ax.set_aspect("equal")
    ax.set_xlabel("temperatura real (°C)"); ax.set_ylabel("temperatura predicha (°C)")
    ax.set_title(titulo)
    ax.text(0.04, 0.96, f"error {rmse(y, p):.2f} °C\nR² {r2(y, p):.3f}", transform=ax.transAxes,
            va="top", fontsize=12, color=color, linespacing=1.4)


# --- los algoritmos y sus diales ---
ALGORITMOS = ["Regresión lineal", "Ridge", "Árbol de decisión", "Random forest", "XGBoost",
              "k-NN (MAT)", "SVM", "Red neuronal"]
# clave, etiqueta, tipo, valor por defecto, mínimo, máximo (en los «log», potencias de 10)
DIALES = {
    "Ridge": [("alpha", "λ, penalización", "log", 1.0, -3, 4)],
    "Árbol de decisión": [("max_depth", "profundidad", "int", 6, 1, 30),
                          ("min_samples_leaf", "muestras mínimas por hoja", "int", 1, 1, 100)],
    "Random forest": [("n_estimators", "nº de árboles", "int", 100, 1, 300),
                      ("max_depth", "profundidad máxima", "int", 30, 1, 30)],
    "XGBoost": [("n_estimators", "nº de árboles", "int", 300, 10, 1000),
                ("learning_rate", "tasa de aprendizaje", "log", 0.1, -3, 0),
                ("max_depth", "profundidad", "int", 6, 1, 12)],
    "k-NN (MAT)": [("n_neighbors", "k, nº de vecinos", "int", 5, 1, 150)],
    "SVM": [("C", "C, tolerancia al error", "log", 10.0, -1, 3)],
    "Red neuronal": [("capas", "capas ocultas", "int", 2, 1, 4),
                     ("neuronas", "neuronas por capa", "int", 32, 2, 128),
                     ("learning_rate_init", "tasa de aprendizaje", "log", 0.001, -4, -1)],
}


def crear_modelo(algoritmo, **hp):
    d = {c: v for c, _, _, v, _, _ in DIALES.get(algoritmo, [])}
    d.update(hp)
    if algoritmo == "Regresión lineal":
        return LinearRegression()
    if algoritmo == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=d["alpha"]))
    if algoritmo == "Árbol de decisión":
        return DecisionTreeRegressor(max_depth=int(d["max_depth"]), min_samples_leaf=int(d["min_samples_leaf"]),
                                     random_state=0)
    if algoritmo == "Random forest":
        return RandomForestRegressor(n_estimators=int(d["n_estimators"]), max_depth=int(d["max_depth"]),
                                     n_jobs=-1, random_state=0)
    if algoritmo == "XGBoost":
        if XGBRegressor is None:
            return HistGradientBoostingRegressor(max_iter=int(d["n_estimators"]), learning_rate=d["learning_rate"],
                                                 max_depth=int(d["max_depth"]), random_state=0)
        return XGBRegressor(n_estimators=int(d["n_estimators"]), learning_rate=d["learning_rate"],
                            max_depth=int(d["max_depth"]), n_jobs=2, tree_method="hist", random_state=0)
    if algoritmo == "k-NN (MAT)":
        return KNeighborsRegressor(n_neighbors=int(d["n_neighbors"]))
    if algoritmo == "SVM":
        return make_pipeline(StandardScaler(), SVR(C=d["C"]))
    if algoritmo == "Red neuronal":
        return make_pipeline(StandardScaler(), MLPRegressor(
            hidden_layer_sizes=(int(d["neuronas"]),) * int(d["capas"]), learning_rate_init=d["learning_rate_init"],
            max_iter=500, early_stopping=True, random_state=0))
    raise ValueError(algoritmo)


# --- la tabla donde se apunta cada modelo que se entrena ---
REGISTRO = {}


def apuntar(donde, algoritmo, predictores, diales, e_train, e_test, r2_test):
    REGISTRO[(algoritmo, predictores, diales)] = dict(
        celda=donde, algoritmo=algoritmo, predictores=predictores, hiperparámetros=diales,
        error_entrenamiento=round(e_train, 2), error_datos_nuevos=round(e_test, 2), R2_datos_nuevos=round(r2_test, 3))


print("Listo.", "XGBoost disponible." if XGBRegressor else "Sin XGBoost: se usará el gradient boosting de scikit-learn.")

---
# 1. Datos

**El conjunto:** 4205 sitios de sedimento superficial marino de todo el océano. En cada uno se han contado los foraminíferos planctónicos (50 especies, como fracciones que suman 1) y se le ha puesto la temperatura media anual del agua en superficie (World Ocean Atlas). Censos de ForCenS (Siccha y Kucera, 2017).

**Pero este fichero viene sucio.** Le hemos metido errores a propósito, de los que aparecen en cualquier base de datos real: códigos de dato ausente, unidades cambiadas, filas repetidas, sitios en tierra… Además, el conjunto original trae algunos problemas propios. Hay que encontrarlos antes de entrenar nada.

## Exploración y limpieza

### 1.1 Cargar los datos

In [ ]:
#@title 1.1 Cargar el fichero
sucio = pd.read_csv(ruta("datos/forams_sst_sucio.csv.gz"))
SITIO = ["Latitude", "Longitude", "Ocean", "sst"]
ESPECIES = [c for c in sucio.columns if c not in SITIO]
PACH = "Neogloboquadrina_pachyderma"

display(Markdown(f"**{len(sucio)} filas** (una por sitio) y **{sucio.shape[1]} columnas**: "
                 f"{len(SITIO)} que describen el sitio y {len(ESPECIES)} de especies."))
muestra = ["Neogloboquadrina_pachyderma", "Globigerina_bulloides", "Globigerinoides_white", "Trilobatus_sacculifer"]
vista = sucio[SITIO + muestra].head(8).rename(columns={e: corto(e) for e in muestra})
display(vista.style.format(precision=3).background_gradient(cmap=LinearSegmentedColormap.from_list("a", ["white", "#9CC3D0"]),
                                                             subset=[corto(e) for e in muestra]))

Cada fila es un sitio. `sst` es la temperatura (°C), el **objetivo** que se quiere predecir. Las columnas de especie son la fracción de cada una en la asociación: 0,25 quiere decir que un 25 % de los individuos contados eran de esa especie.

### 1.2 Un primer vistazo

**Qué hacer:** ejecutar la celda.

**Qué mirar:** antes de calcular nada, dibujar. Un mapa y un histograma ya delatan muchos errores.

In [ ]:
#@title 1.2 Mapa e histograma de la temperatura
fig = plt.figure(figsize=(14, 4.6))
ax = fig.add_axes([0.03, 0.1, 0.58, 0.8])
fondo_mapa(ax, lat=(-90, max(95, sucio["Latitude"].max() + 8)))
ax.axhline(90, color=ROJO, lw=1, ls=(0, (4, 3))); ax.axhline(-90, color=ROJO, lw=1, ls=(0, (4, 3)))
ax.text(-178, 92, "latitud 90°: el polo", color=ROJO, fontsize=9, va="bottom")
ax.set_ylim(min(-95, sucio["Latitude"].min() - 10), max(100, sucio["Latitude"].max() + 10))
ax.scatter(sucio["Longitude"], sucio["Latitude"], s=5, color=AZUL, alpha=0.6, lw=0)
ax.set_title("Dónde están los sitios")

ax = fig.add_axes([0.68, 0.16, 0.3, 0.7])
v = sucio["sst"].dropna()
ax.hist(v, bins=120, color=AZUL)
ax.set_xlabel("sst (°C)"); ax.set_ylabel("nº de sitios")
ax.set_title("Temperatura del agua")
plt.show()

display(Markdown("**El resumen estadístico de la temperatura y las coordenadas:**"))
display(sucio[["sst", "Latitude", "Longitude"]].describe().T[["count", "mean", "std", "min", "max"]]
        .style.format(precision=1))

**Qué se ve:**

- En el mapa hay puntos **por encima del polo** y algunos que parecen estar **en tierra**.
- El histograma **no se puede leer**: casi todos los sitios se aprietan en una sola barra. Eso pasa cuando hay valores absurdamente grandes o pequeños.
- El resumen lo confirma: una temperatura **mínima de −999** y una **máxima de más de 300 °C**.

Un histograma ilegible ya es una alarma. Toca revisar fila a fila con controles sistemáticos.

### 1.3 Los controles, uno a uno

Diez preguntas que se le pueden hacer a cualquier tabla de este tipo. Las cinco primeras valen para cualquier dato; las demás usan lo que se sabe del sistema (geografía y ecología de los foraminíferos).

| Control | Qué busca |
|---|---|
| Huecos en la temperatura | Sitios sin objetivo: no sirven para entrenar |
| Huecos en las especies | Celdas vacías en las fracciones |
| Filas duplicadas | La misma muestra dos veces: pesa doble y se cuela a la vez en entrenamiento y en prueba |
| Temperatura fuera de rango | El océano va de −2 a 32 °C en superficie |
| Especies que no suman 1 | Fracciones mal calculadas o porcentajes |
| Fracciones negativas | Imposibles |
| Coordenadas imposibles | Latitud fuera de ±90° |
| Sitios en tierra | Un sedimento marino en tierra firme |
| *N. pachyderma* en aguas cálidas | Especie polar con más del 50 % en aguas de más de 15 °C: identificación o etiqueta dudosa |
| Muestras con 1 o 2 especies | Fuera de las aguas polares, casi siempre son pocos individuos contados o disolución |

**Qué hacer:** ejecutar la celda.

In [ ]:
#@title 1.3 Pasar los controles
def revisar(t):
    tot = t[ESPECIES].sum(axis=1)
    frac_pach = t[PACH] / tot.where(tot > 0)
    riqueza = (t[ESPECIES] > 0).sum(axis=1)
    sst_ok = t["sst"].between(-2.5, 32)
    coord_ok = t["Latitude"].between(-90, 90) & t["Longitude"].between(-180, 180)
    return {
        "Huecos en la temperatura": t["sst"].isna(),
        "Huecos en las especies": t[ESPECIES].isna().any(axis=1),
        "Filas duplicadas": t.duplicated(),
        "Temperatura fuera de rango": t["sst"].notna() & ~sst_ok,
        "Especies que no suman 1": t[ESPECIES].notna().all(axis=1) & ~tot.between(0.99, 1.01),
        "Fracciones negativas": (t[ESPECIES] < 0).any(axis=1),
        "Coordenadas imposibles": ~coord_ok,
        "Sitios en tierra": pd.Series(coord_ok.values & en_tierra(t["Longitude"], t["Latitude"]), index=t.index),
        "N. pachyderma en aguas cálidas": (frac_pach > 0.5) & sst_ok & (t["sst"] > 15),
        "Muestras con 1 o 2 especies": (riqueza <= 2) & (frac_pach < 0.5),
    }


def semaforo(ax, controles, titulo):
    nombres = list(controles)[::-1]
    n = [int(controles[k].sum()) for k in nombres]
    ax.barh(nombres, n, color=[VERDE if x == 0 else ROJO for x in n], height=0.65)
    for i, x in enumerate(n):
        ax.text(x + 0.4, i, str(x) if x else "0  ✓", va="center", fontsize=11, color=TINTA if x else VERDE)
    ax.set_xlim(0, max(max(n), 5) * 1.15); ax.grid(axis="y", visible=False)
    ax.set_xlabel("filas señaladas"); ax.set_title(titulo)
    ax.tick_params(axis="y", length=0)


CONTROLES = revisar(sucio)
malas = pd.concat(CONTROLES, axis=1).any(axis=1)
fig, ax = plt.subplots(figsize=(9, 4.6))
semaforo(ax, CONTROLES, "Cuántas filas señala cada control")
plt.show()
display(Markdown(f"En total, **{int(malas.sum())} filas** de {len(sucio)} tienen al menos un problema."))

### 1.4 Ver cada problema

**Qué mirar:** en rojo, las filas que señala cada control. Casi todos los errores se reconocen a simple vista una vez se dibuja la variable adecuada.

In [ ]:
#@title 1.4 Los números
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"wspace": 0.28})
r = np.random.default_rng(0)

ax = axs[0]
v = sucio["sst"]; mal = CONTROLES["Temperatura fuera de rango"]
ax.axvspan(-2.5, 32, color=VERDE, alpha=0.12, lw=0)
ax.scatter(v[~mal], r.uniform(0, 1, (~mal).sum()), s=4, color=AZUL, alpha=0.4, lw=0)
ax.scatter(v[mal], r.uniform(0.1, 0.9, mal.sum()), s=40, color=ROJO, zorder=3)
ax.set_xscale("symlog", linthresh=50); ax.set_yticks([]); ax.grid(False)
ax.set_xticks([-1000, -100, 0, 30, 300]); ax.set_xticklabels(["−1000", "−100", "0", "30", "300"])
ax.set_xlim(-2500, 600)
for x, t in ((-999, "−999\ncódigo de\ndato ausente"), (300, "≈ 300\n¿kelvin?"), (45, "45\nimposible")):
    if ((v - x).abs() < 20).any():
        ax.text(x, 1.05, t, ha="center", va="bottom", fontsize=9, color=ROJO, linespacing=1.1)
ax.set_ylim(-0.05, 1.5); ax.set_xlabel("sst (°C), escala comprimida en los extremos")
ax.set_title("Temperatura")

ax = axs[1]
tot = sucio[ESPECIES].sum(axis=1); mal = CONTROLES["Especies que no suman 1"]
ax.scatter(tot[~mal], r.uniform(0, 1, (~mal).sum()), s=4, color=AZUL, alpha=0.4, lw=0)
ax.scatter(tot[mal], r.uniform(0.1, 0.9, mal.sum()), s=40, color=ROJO, zorder=3)
ax.set_xscale("log"); ax.set_yticks([]); ax.grid(False)
ax.set_xticks([1, 10, 100]); ax.set_xticklabels(["1", "10", "100"])
ax.text(100, 1.05, "100\nporcentajes, no\nfracciones", ha="center", va="bottom", fontsize=9, color=ROJO, linespacing=1.1)
ax.text(1.3, 1.05, "1,3", ha="center", va="bottom", fontsize=9, color=ROJO)
ax.set_ylim(-0.05, 1.5); ax.set_xlabel("suma de las 50 especies")
ax.set_title("Suma de cada fila")

ax = axs[2]
mn = sucio[ESPECIES].min(axis=1); mal = CONTROLES["Fracciones negativas"]
ax.scatter(mn[~mal], r.uniform(0, 1, (~mal).sum()), s=4, color=AZUL, alpha=0.4, lw=0)
ax.scatter(mn[mal], r.uniform(0.1, 0.9, mal.sum()), s=40, color=ROJO, zorder=3)
ax.axvline(0, color=TINTA, lw=1); ax.set_yticks([]); ax.grid(False); ax.set_ylim(-0.05, 1.5)
ax.set_xlabel("valor más bajo de la fila"); ax.set_title("Fracciones negativas")
plt.show()

In [ ]:
#@title 1.4 El sitio y la ecología
fig = plt.figure(figsize=(15, 4.6))
ax = fig.add_axes([0.02, 0.1, 0.5, 0.8])
fondo_mapa(ax)
ax.set_ylim(-80, 90)
mal = CONTROLES["Sitios en tierra"]
ax.scatter(sucio["Longitude"], sucio["Latitude"], s=4, color=AZUL, alpha=0.35, lw=0)
ax.scatter(sucio.loc[mal, "Longitude"], sucio.loc[mal, "Latitude"], s=60, color=ROJO, edgecolor="white", zorder=3)
fuera = sucio[CONTROLES["Coordenadas imposibles"]]
ax.set_title(f"Sitios en tierra (rojo) · {len(fuera)} más con latitud imposible, fuera del mapa")

ok = sucio["sst"].between(-2.5, 32)
frac = sucio[PACH] / sucio[ESPECIES].sum(axis=1)
ax = fig.add_axes([0.57, 0.16, 0.19, 0.7])
mal = CONTROLES["N. pachyderma en aguas cálidas"]
ax.scatter(sucio.loc[ok & ~mal, "sst"], frac[ok & ~mal], s=5, color=AZUL, alpha=0.4, lw=0)
ax.scatter(sucio.loc[mal, "sst"], frac[mal], s=45, color=ROJO, zorder=3)
ax.set_xlabel("sst (°C)"); ax.set_ylabel("fracción de N. pachyderma")
ax.set_title("Una especie polar")

ax = fig.add_axes([0.81, 0.16, 0.18, 0.7])
ric = (sucio[ESPECIES] > 0).sum(axis=1)
mal = CONTROLES["Muestras con 1 o 2 especies"]
ax.scatter(sucio.loc[ok & ~mal, "sst"], ric[ok & ~mal], s=5, color=AZUL, alpha=0.4, lw=0)
ax.scatter(sucio.loc[mal, "sst"], ric[mal], s=45, color=ROJO, zorder=3)
ax.set_xlabel("sst (°C)"); ax.set_ylabel("nº de especies presentes")
ax.set_title("Riqueza")
plt.show()

**Lo que dicen las figuras:**

- **−999** es un código clásico de «sin dato» que alguien no tradujo. **≈300** son kelvin (300 K son unos 27 °C).
- Las filas que **suman 100** son porcentajes donde el resto son fracciones.
- *N. pachyderma* vive en aguas polares: con un 70 % en aguas de 27 °C, lo más probable es una etiqueta o una identificación equivocada. Los puntos azules de la izquierda, los sitios fríos con casi un 100 % de *pachyderma*, son reales.
- Un sitio tropical con una sola especie (*Orbulina universa* al 100 %) suele ser una muestra con muy pocos individuos.

### 1.5 Corregir lo que tiene arreglo y quitar el resto

Algunos errores se pueden **corregir** sin inventar nada: kelvin a grados, porcentajes a fracciones, latitud y longitud cambiadas de sitio. El resto se **quita**: no hay forma honesta de saber cuál era el valor bueno.

**Qué hacer:** ejecutar la celda.

In [ ]:
#@title 1.5 Limpiar
limpio = sucio.copy()
arreglos = {}
k = limpio["sst"].between(250, 320)
limpio.loc[k, "sst"] = limpio.loc[k, "sst"] - 273.15; arreglos["kelvin → °C"] = int(k.sum())
tot = limpio[ESPECIES].sum(axis=1)
p = tot.between(90, 110)
limpio.loc[p, ESPECIES] = limpio.loc[p, ESPECIES].div(100); arreglos["porcentajes → fracciones"] = int(p.sum())
c = ~limpio["Latitude"].between(-90, 90) & limpio["Latitude"].between(-180, 180) & limpio["Longitude"].between(-90, 90)
limpio.loc[c, ["Latitude", "Longitude"]] = limpio.loc[c, ["Longitude", "Latitude"]].values
arreglos["latitud y longitud cambiadas"] = int(c.sum())

quitar = pd.concat(revisar(limpio), axis=1).any(axis=1)
limpio = limpio[~quitar].reset_index(drop=True)
DESPUES = revisar(limpio)

fig, axs = plt.subplots(1, 2, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1, 1.25], "wspace": 0.55})
ax = axs[0]
pasos = [("al principio", len(sucio), TINTA), ("corregidas", sum(arreglos.values()), ORO),
         ("quitadas", int(quitar.sum()), ROJO), ("quedan", len(limpio), VERDE)]
ax.barh([p[0] for p in pasos][::-1], [p[1] for p in pasos][::-1], color=[p[2] for p in pasos][::-1], height=0.6)
for i, (_, n, _) in enumerate(pasos[::-1]):
    ax.text(n + 40, i, f"{n}", va="center", fontsize=12)
ax.set_xlim(0, len(sucio) * 1.2); ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0)
ax.set_xlabel("filas"); ax.set_title("Qué ha pasado con las filas")
semaforo(axs[1], DESPUES, "Los controles, otra vez")
plt.show()
display(Markdown("**Corregidas:** " + ", ".join(f"{k} ({v})" for k, v in arreglos.items()) +
                 f". **Quitadas:** {int(quitar.sum())} filas. Queda una tabla de **{len(limpio)} sitios** sin ningún aviso."))

**Contar las filas después de cada paso** y compararlas con lo esperado es la costumbre más útil de todo el análisis de datos. Un filtro que se come la mitad de la tabla no da ningún error.

### 1.6 Cómo es el conjunto limpio

**Qué mirar:** ahora sí, el mapa y el histograma se pueden leer. El reparto de los sitios no es uniforme: el Atlántico Norte está mucho mejor muestreado que el Pacífico Sur.

In [ ]:
#@title 1.6 Mapa, histograma y latitud
fig = plt.figure(figsize=(15, 4.8))
ax = fig.add_axes([0.02, 0.1, 0.5, 0.82])
fondo_mapa(ax)
sc = ax.scatter(limpio["Longitude"], limpio["Latitude"], c=limpio["sst"], cmap=SST_CMAP, s=7, lw=0, vmin=-2, vmax=30)
cb = fig.colorbar(sc, ax=ax, fraction=0.025, pad=0.01); cb.outline.set_visible(False); cb.set_label("sst (°C)")
ax.set_title(f"Los {len(limpio)} sitios, coloreados por temperatura")

ax = fig.add_axes([0.63, 0.16, 0.15, 0.72])
n, b, parches = ax.hist(limpio["sst"], bins=32, range=(-2, 30))
for x, pa in zip(b, parches):
    pa.set_facecolor(SST_CMAP((x + 2) / 32))
ax.set_xlabel("sst (°C)"); ax.set_ylabel("nº de sitios"); ax.set_title("Temperatura")

ax = fig.add_axes([0.82, 0.16, 0.17, 0.72])
ax.scatter(limpio["sst"], limpio["Latitude"], c=limpio["sst"], cmap=SST_CMAP, s=5, lw=0, vmin=-2, vmax=30)
ax.set_xlabel("sst (°C)"); ax.set_ylabel("latitud"); ax.set_title("Temperatura y latitud")
plt.show()

**Fijarse en el histograma:** hay muchos más sitios cálidos que fríos. Un modelo aprende mejor donde tiene más ejemplos, así que se puede esperar más error en las aguas frías.

## Selección de predictores

### 1.7 Qué especies cuentan

Antes de meter 50 columnas en un modelo, conviene saber cuáles tienen información. Tres preguntas: en cuántos sitios aparece cada especie, qué abundancia llega a tener y cuánto se relaciona con la temperatura.

**Qué hacer:** ejecutar la celda.

In [ ]:
#@title 1.7 Presencia, abundancia y relación con la temperatura
presencia = (limpio[ESPECIES] > 0).mean() * 100
maximo = limpio[ESPECIES].max() * 100
rho = limpio[ESPECIES].corrwith(limpio["sst"], method="spearman").fillna(0)

fig, axs = plt.subplots(1, 3, figsize=(15, 9), gridspec_kw={"wspace": 0.75})
orden = rho.sort_values().index
nombres = [corto(e) for e in orden]
axs[0].barh(nombres, presencia[orden], color=AZUL, height=0.7)
axs[0].set_xlabel("% de sitios donde aparece"); axs[0].set_title("Presencia")
axs[1].barh(nombres, maximo[orden], color=TINTA, height=0.7)
axs[1].set_xlabel("abundancia máxima (%)"); axs[1].set_title("Abundancia máxima")
axs[2].barh(nombres, rho[orden], color=[AZUL if x < 0 else ROJO for x in rho[orden]], height=0.7)
axs[2].axvline(0, color=TINTA, lw=0.8)
axs[2].set_xlabel("correlación con la temperatura (Spearman)"); axs[2].set_title("Relación con la temperatura")
for ax in axs:
    ax.tick_params(axis="y", labelsize=8.5, length=0); ax.grid(axis="y", visible=False); ax.margins(y=0.01)
plt.show()

**Arriba**, las especies de aguas cálidas (correlación positiva, en rojo). **Abajo**, las de aguas frías (en azul). **En medio**, las que apenas dicen nada de la temperatura, a menudo raras: aparecen en pocos sitios y con abundancias mínimas.

La correlación solo mide relaciones que siempre suben o siempre bajan. Una especie de aguas templadas, que es abundante a 15 °C y escasa a 0 y a 28, puede tener correlación casi nula y aun así ser muy informativa. Por eso conviene mirar las curvas.

In [ ]:
#@title 1.7 Curvas de respuesta de cinco especies clave
CLAVE = ["Neogloboquadrina_pachyderma", "Neogloboquadrina_incompta", "Globoconella_inflata",
         "Globigerinoides_white", "Trilobatus_sacculifer"]
fig, axs = plt.subplots(1, 5, figsize=(16, 3.4), sharey=True, gridspec_kw={"wspace": 0.08})
tramos = np.arange(-2, 31, 2)
for ax, e, col in zip(axs, CLAVE, ["#16405A", AZUL, "#B8862B", "#D9734E", ROJO]):
    ax.scatter(limpio["sst"], limpio[e] * 100, s=3, color=GRIS, alpha=0.25, lw=0)
    centro = (tramos[:-1] + tramos[1:]) / 2
    media = [limpio.loc[limpio["sst"].between(a, b), e].mean() * 100 for a, b in zip(tramos[:-1], tramos[1:])]
    ax.plot(centro, media, color=col, lw=2.8)
    ax.set_title(corto(e), style="italic"); ax.set_xlabel("sst (°C)")
axs[0].set_ylabel("abundancia (%)")
plt.show()

Cinco especies, cinco óptimos de temperatura distintos: de polar a tropical. *G. inflata* y *N. incompta* son las de aguas templadas que la correlación infravalora.

### 1.8 Redundancia entre especies

Dos predictores muy correlacionados entre sí aportan casi la misma información. No estorba a todos los modelos por igual, pero conviene saberlo, sobre todo para interpretar después qué ha usado el modelo.

In [ ]:
#@title 1.8 Correlación entre las 15 especies más abundantes
top = limpio[ESPECIES].mean().sort_values(ascending=False).index[:15]
cm = limpio[top].corr(method="spearman").values
fig, ax = plt.subplots(figsize=(8.5, 7.2))
im = ax.imshow(cm, cmap=LinearSegmentedColormap.from_list("d", [AZUL, "white", ROJO]), vmin=-1, vmax=1)
ax.set_xticks(range(15)); ax.set_yticks(range(15))
ax.set_xticklabels([corto(e) for e in top], rotation=60, ha="right", fontsize=9)
ax.set_yticklabels([corto(e) for e in top], fontsize=9)
ax.grid(False); ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, fraction=0.04, pad=0.02); cb.outline.set_visible(False); cb.set_label("correlación de Spearman")
ax.set_title("Qué especies van juntas")
plt.show()

Los bloques rojos son especies que suelen aparecer juntas (por ejemplo, las tropicales entre sí). Los azules, especies que se excluyen: donde hay una, no hay la otra.

Ojo con un detalle de este tipo de datos: **las fracciones suman 1**. Si una especie sube, las demás bajan a la fuerza. Parte de las correlaciones negativas viene de ahí y no de la ecología.

## Ingeniería

### 1.9 Transformar y quitar las especies raras

Dos decisiones clásicas en funciones de transferencia:

- **Transformación.** Con la **raíz cuadrada** de las fracciones (en este caso equivale a la transformación de Hellinger), las especies dominantes pesan menos y las minoritarias cuentan algo más. Es la habitual para comparar asociaciones.
- **Especies raras.** Las que nunca llegan a un porcentaje mínimo en ningún sitio aportan sobre todo ruido.

**Qué hacer:** elegir en los controles de la derecha y ejecutar la celda. Si se cambia algo aquí, todo lo que se entrene después lo usará.

In [ ]:
#@title 1.9 Transformación y especies raras
transformacion = "Raíz cuadrada (Hellinger)"  #@param ["Ninguna", "Raíz cuadrada (Hellinger)"]
umbral_raras = 1  #@param {type:"slider", min:0, max:10, step:1}

maximo = limpio[ESPECIES].max() * 100
USADAS = [e for e in ESPECIES if maximo[e] >= umbral_raras and maximo[e] > 0]


def transformar(tabla):
    return np.sqrt(tabla.clip(lower=0)) if transformacion.startswith("Raíz") else tabla


fig, axs = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [1, 1, 1.1], "wspace": 0.3})
e = "Globigerina_bulloides"
axs[0].hist(limpio[e], bins=40, color=GRIS); axs[0].set_title(f"{corto(e)}, sin transformar")
axs[0].set_xlabel("fracción"); axs[0].set_ylabel("nº de sitios")
axs[1].hist(transformar(limpio[[e]])[e], bins=40, color=AZUL)
axs[1].set_title(f"{corto(e)}, " + ("raíz cuadrada" if transformacion.startswith("Raíz") else "sin transformar"))
axs[1].set_xlabel("valor transformado")
fuera = len(ESPECIES) - len(USADAS)
axs[2].barh(["se quitan", "se usan"], [fuera, len(USADAS)], color=[ROJO, VERDE], height=0.55)
for i, n in enumerate([fuera, len(USADAS)]):
    axs[2].text(n + 0.5, i, str(n), va="center", fontsize=13)
axs[2].set_xlim(0, 58); axs[2].grid(axis="y", visible=False); axs[2].tick_params(axis="y", length=0)
axs[2].set_title(f"Especies que nunca pasan del {umbral_raras} %")
plt.show()
display(Markdown(f"Quedan **{len(USADAS)} especies** como predictores."))

### 1.10 Reducir dimensiones: PCA

Con 50 especies no se puede dibujar cada sitio. El **análisis de componentes principales** resume la tabla en unos pocos ejes que recogen la mayor parte de la variación. Si los sitios se ordenan por temperatura a lo largo del primer eje, la temperatura es lo que más estructura la asociación, y eso es una buena noticia para el modelo.

**Qué hacer:** ejecutar la celda.

In [ ]:
#@title 1.10 PCA de las asociaciones
Z = transformar(limpio[USADAS])
pca = PCA(n_components=10).fit(Z)
P = pca.transform(Z)
var = pca.explained_variance_ratio_ * 100

fig, axs = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.6, 1], "wspace": 0.25})
ax = axs[0]
sc = ax.scatter(P[:, 0], P[:, 1], c=limpio["sst"], cmap=SST_CMAP, s=7, lw=0, vmin=-2, vmax=30)
cb = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.01); cb.outline.set_visible(False); cb.set_label("sst (°C)")
cargas = pca.components_[:2].T
fuertes = np.argsort(-np.hypot(*cargas.T))[:7]
esc = np.abs(P[:, :2]).max() * 0.6 / np.abs(cargas[fuertes]).max()
for i in fuertes:
    x, y = cargas[i] * esc
    ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=TINTA, lw=1.2))
    ax.text(x * 1.12, y * 1.12, corto(USADAS[i]), fontsize=9.5, style="italic", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
ax.set_xlabel(f"eje 1 ({var[0]:.0f} % de la variación)"); ax.set_ylabel(f"eje 2 ({var[1]:.0f} %)")
ax.set_title("Cada punto es un sitio; las flechas, las especies que más pesan")

ax = axs[1]
ax.bar(range(1, 11), var, color=AZUL)
ax.plot(range(1, 11), np.cumsum(var), color=ROJO, marker="o", ms=5)
ax.set_xticks(range(1, 11)); ax.set_xlabel("eje"); ax.set_ylabel("% de la variación")
ax.text(10, np.cumsum(var)[-1] + 2, f"{np.cumsum(var)[-1]:.0f} % acumulado", ha="right", color=ROJO)
ax.set_ylim(0, 105); ax.set_title("Cuánto recoge cada eje")
plt.show()
r1 = np.corrcoef(P[:, 0], limpio["sst"])[0, 1]
display(Markdown(f"El primer eje recoge un **{var[0]:.0f} %** de la variación y su correlación con la temperatura es **{abs(r1):.2f}**".replace(".", ",") + "."))

El PCA no sabe nada de la temperatura: solo ha mirado las especies. Que el color se ordene a lo largo del primer eje quiere decir que la temperatura es lo que más estructura las asociaciones.

---
# 2. Algoritmo

### 2.1 La carta de algoritmos

Los mismos de la diapositiva. Todos están disponibles en el resto del cuaderno.

| Familia | Algoritmo | En una frase |
|---|---|---|
| Lineales | **Regresión lineal** | Una suma de especies con un peso cada una. La referencia que hay que batir |
| | **Ridge** | Lo mismo, con los pesos frenados por una penalización λ |
| Árboles | **Árbol de decisión** | Preguntas encadenadas del tipo «¿*pachyderma* > 0,3?» |
| | **Random forest** | Cientos de árboles con muestras al azar, promediados |
| | **XGBoost** | Árboles en serie, cada uno corrige al anterior |
| Similitud y redes | **k-NN (MAT)** | La media de los *k* sitios con la asociación más parecida. Es la técnica de los análogos modernos |
| | **SVM** | Una curva con un margen de tolerancia al error |
| | **Red neuronal** | Capas de sumas ponderadas y funciones no lineales |

Con la transformación de raíz cuadrada, la distancia que usa el k-NN es la **distancia de cuerda** (*chord distance*), la misma del MAT clásico.

### 2.2 ¿Y si no separamos los datos?

La tentación: entrenar el modelo con todos los sitios y luego mirar qué tal acierta… **en esos mismos sitios**.

**Qué hacer:** elegir un algoritmo a la derecha y ejecutar. Probar primero con el árbol de decisión.

In [ ]:
#@title 2.2 Entrenar y evaluar con los mismos datos
algoritmo = "Árbol de decisión"  #@param ["Regresión lineal", "Ridge", "Árbol de decisión", "Random forest", "XGBoost", "k-NN (MAT)", "SVM", "Red neuronal"]
PREDICTORES = ["Todas las especies", "Solo las 5 especies clave", "Todas menos N. pachyderma",
               "Especies + latitud y longitud", "Solo la latitud"]


def matriz(predictores, tabla=None):
    t = limpio if tabla is None else tabla
    T = transformar(t[USADAS]).fillna(0)
    if predictores == "Todas las especies":
        X = T
    elif predictores == "Solo las 5 especies clave":
        X = T[[e for e in CLAVE if e in USADAS]]
    elif predictores == "Todas menos N. pachyderma":
        X = T.drop(columns=[PACH], errors="ignore")
    elif predictores == "Especies + latitud y longitud":
        X = T.assign(lat=t["Latitude"], lon=t["Longitude"])
    elif predictores == "Solo la latitud":
        X = pd.DataFrame({"distancia al ecuador": t["Latitude"].abs()})
    return X.values, t["sst"].values


X, y = matriz("Todas las especies")
hp = {"max_depth": 30} if algoritmo == "Árbol de decisión" else {"n_neighbors": 1} if algoritmo == "k-NN (MAT)" else {}
m = crear_modelo(algoritmo, **hp).fit(X, y)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
obs_pred(ax, y, m.predict(X), ROJO, f"{algoritmo}: entrenado y evaluado\ncon los mismos {len(y)} sitios")
plt.show()

Con el árbol sin límite de profundidad (o el k-NN con un solo vecino), el error es casi **cero** y el R² roza **1,000**. Parece un modelo perfecto.

No lo es: **se ha aprendido de memoria los sitios**, incluido su ruido. Preguntarle por los sitios que ya ha visto es como poner en el examen las mismas preguntas del libro de ejercicios. Para saber si ha aprendido algo hay que preguntarle por sitios que **no ha visto**.

### 2.3 El split: entrenamiento y prueba

Se separa una parte de los sitios **antes de entrenar** y se esconde. El modelo aprende con el resto, y al final se le evalúa con la parte escondida.

**Qué hacer:** elegir qué fracción se reserva para la prueba y ejecutar. La separación que salga aquí es la que se usa en toda la sección 3.

In [ ]:
#@title 2.3 Separar entrenamiento y prueba
fraccion_prueba = 0.25  #@param {type:"slider", min:0.1, max:0.5, step:0.05}
ENTRENO, PRUEBA = train_test_split(np.arange(len(limpio)), test_size=fraccion_prueba, random_state=42)

fig = plt.figure(figsize=(15, 5))
ax = fig.add_axes([0.02, 0.1, 0.42, 0.8])
fondo_mapa(ax)
ax.scatter(limpio.loc[ENTRENO, "Longitude"], limpio.loc[ENTRENO, "Latitude"], s=5, color=AZUL, alpha=0.6, lw=0, label="entrenamiento")
ax.scatter(limpio.loc[PRUEBA, "Longitude"], limpio.loc[PRUEBA, "Latitude"], s=9, color=ORO, lw=0, label="prueba")
ax.legend(loc="lower left", markerscale=2.5, fontsize=11)
ax.set_title(f"{len(ENTRENO)} sitios para entrenar, {len(PRUEBA)} escondidos para la prueba")

m = crear_modelo(algoritmo, **hp).fit(X[ENTRENO], y[ENTRENO])
ax = fig.add_axes([0.5, 0.12, 0.22, 0.76]); obs_pred(ax, y[ENTRENO], m.predict(X[ENTRENO]), AZUL, "Con los sitios de entrenamiento")
ax = fig.add_axes([0.77, 0.12, 0.22, 0.76]); obs_pred(ax, y[PRUEBA], m.predict(X[PRUEBA]), ORO, "Con los sitios escondidos")
plt.show()

**La diferencia entre las dos figuras de la derecha es el sobreajuste.** Con los sitios de entrenamiento, el modelo de la celda 2.2 parece perfecto. Con los escondidos, el error real es varias veces mayor.

Esa cifra, la de los sitios escondidos, es la única que vale para decir lo bien que funciona un modelo.

---
# 3. Entrenamiento

### 3.1 El laboratorio de hiperparámetros

Aquí se juega. Se elige un algoritmo y unos predictores, se mueven los diales y el modelo se reentrena solo.

**Qué mirar:**

- **Entrenamiento (azul) frente a datos nuevos (oro).** Si el error de entrenamiento es mucho menor que el de los datos nuevos, el modelo **sobreajusta**. Si los dos son altos, **infraajusta**: es demasiado simple.
- Cada modelo que se entrena **se apunta** en la tabla final (sección 4.4).

**Algunas cosas que probar:**

- Árbol de decisión: profundidad 1, luego 5, luego 30.
- k-NN: k = 1, luego 5, luego 150.
- Random forest: 1 árbol, luego 10, luego 300.
- Red neuronal: tasa de aprendizaje de 0,0001 y de 0,1.

In [ ]:
#@title 3.1 Laboratorio (los controles aparecen debajo)
estilo = {"description_width": "190px"}
w_alg = w.Dropdown(options=ALGORITMOS, value="Random forest", description="Algoritmo", style=estilo, layout=w.Layout(width="480px"))
w_pred = w.Dropdown(options=PREDICTORES, value="Todas las especies", description="Predictores", style=estilo, layout=w.Layout(width="480px"))
caja = w.VBox()
estado = w.HTML()
salida = w.Output()
DIALES_ACTIVOS = []


def construir_diales(*_):
    DIALES_ACTIVOS.clear()
    hijos = []
    for clave, etiqueta, tipo, defecto, lo, hi in DIALES.get(w_alg.value, []):
        if tipo == "int":
            s = w.IntSlider(value=defecto, min=lo, max=hi, description=etiqueta, continuous_update=False,
                            style=estilo, layout=w.Layout(width="620px"))
        else:
            s = w.FloatLogSlider(value=defecto, base=10, min=lo, max=hi, step=0.25, description=etiqueta,
                                 readout_format=".4g", continuous_update=False, style=estilo, layout=w.Layout(width="620px"))
        s.observe(entrenar, names="value")
        DIALES_ACTIVOS.append((clave, etiqueta, s))
        hijos.append(s)
    caja.children = hijos or [w.HTML("<i>La regresión lineal no tiene hiperparámetros: los pesos salen directamente de los datos.</i>")]


def entrenar(*_):
    hp = {c: s.value for c, _, s in DIALES_ACTIVOS}
    texto = ", ".join(f"{e} = {s.value:.4g}" if isinstance(s.value, float) else f"{e} = {s.value}" for _, e, s in DIALES_ACTIVOS) or "sin diales"
    estado.value = "<i>Entrenando…</i>"
    X, y = matriz(w_pred.value)
    t0 = time.time()
    m = crear_modelo(w_alg.value, **hp).fit(X[ENTRENO], y[ENTRENO])
    pa, pb = m.predict(X[ENTRENO]), m.predict(X[PRUEBA])
    ea, eb = rmse(y[ENTRENO], pa), rmse(y[PRUEBA], pb)
    apuntar("3.1", w_alg.value, w_pred.value, texto, ea, eb, r2(y[PRUEBA], pb))
    if eb > 2.5 and ea > 0.75 * eb:
        diag, color = "infraajusta: falla parecido en los dos, el modelo es demasiado simple", ORO
    elif eb > 1.6 * ea + 0.3:
        diag, color = "señal de sobreajuste: acierta mucho más con lo que ha visto. Lo que cuenta es el error con datos nuevos", ROJO
    else:
        diag, color = "equilibrado: el error con datos nuevos se parece al de entrenamiento", VERDE
    with salida:
        clear_output(wait=True)
        fig = plt.figure(figsize=(15, 4.6))
        a1 = fig.add_axes([0.05, 0.13, 0.25, 0.75]); obs_pred(a1, y[ENTRENO], pa, AZUL, "Entrenamiento")
        a2 = fig.add_axes([0.37, 0.13, 0.25, 0.75]); obs_pred(a2, y[PRUEBA], pb, ORO, "Datos nuevos (prueba)")
        a3 = fig.add_axes([0.72, 0.13, 0.26, 0.75])
        a3.bar(["entrenamiento", "datos nuevos"], [ea, eb], color=[AZUL, ORO], width=0.55)
        for i, v in enumerate([ea, eb]):
            a3.text(i, v + 0.05, f"{v:.2f} °C", ha="center", va="bottom", fontsize=13)
        a3.set_ylim(0, max(6, eb * 1.7)); a3.set_ylabel("error (RMSE, °C)"); a3.grid(axis="x", visible=False)
        a3.set_title(f"{w_alg.value} · {w_pred.value}", fontsize=11.5)
        a3.text(0.03, 0.97, textwrap.fill(diag, 44), transform=a3.transAxes, va="top",
                color=color, fontsize=11, style="italic", linespacing=1.3)
        plt.show()
    estado.value = f"Entrenado en {time.time() - t0:.1f} s · {texto}"


w_alg.observe(lambda *_: (construir_diales(), entrenar()), names="value")
w_pred.observe(entrenar, names="value")
display(w.VBox([w_alg, w_pred, caja, estado, salida]))
construir_diales()
entrenar()

### 3.2 La curva del sobreajuste

En el laboratorio se ve un punto cada vez. Aquí se recorre **todo el dial** de una vez: el error de entrenamiento y el de datos nuevos para cada valor. Es la figura de la diapositiva de Entrenamiento, con datos reales.

**Qué hacer:** elegir el algoritmo y ejecutar. El random forest y la red tardan algo más (unos 30 segundos).

In [ ]:
#@title 3.2 Recorrer un dial entero
curva = "Árbol de decisión · profundidad"  #@param ["Árbol de decisión · profundidad", "k-NN (MAT) · nº de vecinos", "Random forest · profundidad", "Red neuronal · neuronas por capa"]
alg, dial = [s.strip() for s in curva.split("·")]
RECORRIDOS = {
    "Árbol de decisión": ("max_depth", [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 30], {}),
    "k-NN (MAT)": ("n_neighbors", [150, 100, 70, 50, 30, 20, 15, 10, 7, 5, 3, 2, 1], {}),
    "Random forest": ("max_depth", [1, 2, 3, 4, 6, 8, 10, 13, 16, 20, 30], {"n_estimators": 60}),
    "Red neuronal": ("neuronas", [2, 4, 8, 16, 32, 64, 128], {"capas": 2}),
}
clave, valores, fijo = RECORRIDOS[alg]
X, y = matriz("Todas las especies")
ea, eb = [], []
for v in valores:
    m = crear_modelo(alg, **fijo, **{clave: v}).fit(X[ENTRENO], y[ENTRENO])
    ea.append(rmse(y[ENTRENO], m.predict(X[ENTRENO]))); eb.append(rmse(y[PRUEBA], m.predict(X[PRUEBA])))

fig, ax = plt.subplots(figsize=(10, 5))
pos = np.arange(len(valores))
mejor = int(np.argmin(eb))
ax.axvspan(-0.5, mejor - 0.5, color=ORO, alpha=0.08, lw=0); ax.axvspan(mejor + 0.5, len(valores) - 0.5, color=ROJO, alpha=0.06, lw=0)
ax.plot(pos, ea, color=AZUL, lw=2.5, marker="o", label="entrenamiento")
ax.plot(pos, eb, color=ORO, lw=2.5, marker="o", label="datos nuevos")
ax.scatter(mejor, eb[mejor], s=160, color=ORO, edgecolor=TINTA, zorder=5)
ax.annotate(f"mejor: {valores[mejor]} ({eb[mejor]:.2f} °C)", (mejor, eb[mejor]), xytext=(0, 22), textcoords="offset points",
            ha="center", fontsize=11)
techo = max(max(ea), max(eb)) * 1.18
ax.set_ylim(0, techo)
ax.text(-0.3, techo * 0.98, "infraajuste", color="#B8862B", va="top", fontsize=11, style="italic")
ax.text(len(valores) - 0.7, techo * 0.98, "sobreajuste", color=ROJO, va="top", ha="right", fontsize=11, style="italic")
ax.set_xticks(pos); ax.set_xticklabels(valores)
ax.set_xlabel(f"{dial}  →  modelo más complejo"); ax.set_ylabel("error (RMSE, °C)")
ax.set_title(f"{alg}: el error según el dial"); ax.legend(loc="center right")
plt.show()

**Siempre sale lo mismo:** el error de entrenamiento baja sin parar a medida que el modelo se complica, y no avisa de nada. El de datos nuevos baja, toca fondo y vuelve a subir (o se estanca). El punto bueno es el mínimo de la curva de datos nuevos.

En el k-NN el eje va al revés: **menos vecinos es un modelo más complejo**, porque con k = 1 cada predicción depende de un solo sitio.

### 3.3 La red neuronal por dentro

Las redes aprenden **paso a paso** con el descenso del gradiente: en cada vuelta a los datos (una *época*) ajustan un poco los pesos para bajar la función de pérdida. Aquí se ve la pérdida época a época.

**Qué probar:**

- **Tasa de aprendizaje** muy pequeña (0,0001): baja tan despacio que no llega.
- **Tasa muy grande** (0,1): da saltos, la curva oscila.
- **Muchas neuronas y muchas épocas:** la curva de entrenamiento sigue bajando y la de datos nuevos se despega. Es el sobreajuste en directo.

In [ ]:
#@title 3.3 Pérdida por épocas (los controles aparecen debajo)
estilo = {"description_width": "170px"}
r_lr = w.FloatLogSlider(value=0.001, base=10, min=-4, max=-1, step=0.25, description="tasa de aprendizaje", readout_format=".4g", continuous_update=False, style=estilo, layout=w.Layout(width="560px"))
r_n = w.IntSlider(value=32, min=2, max=128, description="neuronas por capa", continuous_update=False, style=estilo, layout=w.Layout(width="560px"))
r_c = w.IntSlider(value=2, min=1, max=4, description="capas ocultas", continuous_update=False, style=estilo, layout=w.Layout(width="560px"))
r_e = w.IntSlider(value=150, min=10, max=400, step=10, description="épocas", continuous_update=False, style=estilo, layout=w.Layout(width="560px"))
salida_red = w.Output()


def red(*_):
    X, y = matriz("Todas las especies")
    esc = StandardScaler().fit(X[ENTRENO])
    xa, xb = esc.transform(X[ENTRENO]), esc.transform(X[PRUEBA])
    m = MLPRegressor(hidden_layer_sizes=(r_n.value,) * r_c.value, learning_rate_init=r_lr.value, random_state=0)
    ea, eb = [], []
    for _ in range(r_e.value):
        m.partial_fit(xa, y[ENTRENO])
        ea.append(rmse(y[ENTRENO], m.predict(xa))); eb.append(rmse(y[PRUEBA], m.predict(xb)))
    with salida_red:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 4.8))
        ep = np.arange(1, len(ea) + 1)
        ax.plot(ep, ea, color=AZUL, lw=2.2, label="entrenamiento")
        ax.plot(ep, eb, color=ORO, lw=2.2, label="datos nuevos")
        ax.set_yscale("log"); ax.set_ylim(0.3, 30)
        ax.set_yticks([0.5, 1, 2, 5, 10, 20]); ax.set_yticklabels(["0.5", "1", "2", "5", "10", "20"])
        ax.set_xlabel("época (vueltas completas a los datos de entrenamiento)"); ax.set_ylabel("error (RMSE, °C, escala log)")
        ax.set_title(f"Al final: {ea[-1]:.2f} °C en entrenamiento, {eb[-1]:.2f} °C con datos nuevos")
        ax.legend()
        plt.show()


for s in (r_lr, r_n, r_c, r_e):
    s.observe(red, names="value")
display(w.VBox([r_lr, r_n, r_c, r_e, salida_red]))
red()

### 3.4 Qué predictores se le dan

El mismo algoritmo con cinco conjuntos de predictores distintos:

- **Todas las especies.**
- **Solo las 5 especies clave** de la celda 1.7.
- **Todas menos *N. pachyderma***, la especie que más pesa en las aguas frías.
- **Especies + latitud y longitud.**
- **Solo la latitud**, sin ninguna especie (la distancia al ecuador).

**Qué hacer:** elegir el algoritmo y ejecutar.

In [ ]:
#@title 3.4 Comparar conjuntos de predictores
algoritmo_pred = "Random forest"  #@param ["Regresión lineal", "Ridge", "Árbol de decisión", "Random forest", "XGBoost", "k-NN (MAT)", "SVM", "Red neuronal"]
res = []
for p in PREDICTORES:
    X, y = matriz(p)
    m = crear_modelo(algoritmo_pred).fit(X[ENTRENO], y[ENTRENO])
    ea, pb = rmse(y[ENTRENO], m.predict(X[ENTRENO])), m.predict(X[PRUEBA])
    res.append((p, ea, rmse(y[PRUEBA], pb)))
    apuntar("3.4", algoritmo_pred, p, "por defecto", ea, rmse(y[PRUEBA], pb), r2(y[PRUEBA], pb))

fig, ax = plt.subplots(figsize=(10, 4.2))
orden = sorted(res, key=lambda r: -r[2])
colores = [ORO if "latitud" in p else AZUL for p, _, _ in orden]
ax.barh([p for p, _, _ in orden], [e for _, _, e in orden], color=colores, height=0.6)
for i, (_, _, e) in enumerate(orden):
    ax.text(e + 0.03, i, f"{e:.2f} °C", va="center", fontsize=12)
ax.set_xlim(0, max(e for _, _, e in orden) * 1.2); ax.grid(axis="y", visible=False); ax.tick_params(axis="y", length=0)
ax.set_xlabel("error con datos nuevos (RMSE, °C)"); ax.set_title(f"{algoritmo_pred}: cuánto cambia según los predictores")
plt.show()

**Tres cosas que salen:**

1. **Solo la latitud ya da un error de unos 2,4 °C.** La temperatura del mar depende sobre todo de la distancia al ecuador, y el modelo no necesita foraminíferos para saberlo. Las especies tienen que batir esa cifra para aportar algo.
2. **Quitar *N. pachyderma* empeora, pero no hunde el modelo.** Otras especies frías llevan parte de la misma información (la redundancia de la celda 1.8).
3. **Añadir latitud y longitud mejora el error.** Parece una buena idea. **No lo es para una función de transferencia:** en el pasado, la latitud de un testigo no cambia y la temperatura sí. Un modelo que se apoya en la geografía predice la temperatura de hoy en ese punto, no la del pasado. La sección 4.3 enseña cómo detectar esta trampa.

### 3.5 ¿Y si no hubiéramos limpiado?

Se entrena el mismo modelo dos veces: con los datos limpios y añadiéndoles las filas con errores de la sección 1. Las dos versiones se evalúan con **los mismos sitios de prueba**, que están limpios.

**Qué hacer:** ejecutar la celda.

In [ ]:
#@title 3.5 Datos sucios frente a limpios
filas_malas = sucio[malas & sucio["sst"].notna() & sucio[ESPECIES].notna().all(axis=1)]
entreno_sucio = pd.concat([limpio.iloc[ENTRENO], filas_malas], ignore_index=True)
Xs, ys = matriz("Todas las especies", entreno_sucio)
X, y = matriz("Todas las especies")
res, pred_sucio = [], {}
for alg in ["Regresión lineal", "Árbol de decisión", "Random forest", "k-NN (MAT)"]:
    el = rmse(y[PRUEBA], crear_modelo(alg).fit(X[ENTRENO], y[ENTRENO]).predict(X[PRUEBA]))
    pred_sucio[alg] = crear_modelo(alg).fit(Xs, ys).predict(X[PRUEBA])
    res.append((alg, el, rmse(y[PRUEBA], pred_sucio[alg])))

fig = plt.figure(figsize=(15, 4.8))
ax = fig.add_axes([0.05, 0.13, 0.55, 0.72])
pos = np.arange(len(res))
ax.bar(pos - 0.2, [r[1] for r in res], width=0.38, color=VERDE, label="entrenado con datos limpios")
ax.bar(pos + 0.2, [r[2] for r in res], width=0.38, color=ROJO, label=f"con {len(filas_malas)} filas erróneas más")
for i, (_, a, b) in enumerate(res):
    ax.text(i - 0.2, a + 0.1, f"{a:.2f}", ha="center", va="bottom"); ax.text(i + 0.2, b + 0.1, f"{b:.2f}", ha="center", va="bottom")
ax.set_ylim(0, max(r[2] for r in res) * 1.3)
ax.set_xticks(pos); ax.set_xticklabels([r[0] for r in res]); ax.grid(axis="x", visible=False)
ax.set_ylabel("error con datos nuevos (RMSE, °C)"); ax.legend(loc="upper left", ncol=2)
ax.set_title(f"{len(filas_malas)} filas malas entre {len(entreno_sucio)}, menos de un 2 %; la prueba es la misma y está limpia")
peor = max(res, key=lambda r: r[2])[0]
ax = fig.add_axes([0.68, 0.13, 0.3, 0.72])
p = pred_sucio[peor]
ax.scatter(y[PRUEBA], p, s=9, color=ROJO, alpha=0.5, lw=0)
ax.axhline(0, color=GRIS, lw=0.8)
ax.set_xlabel("temperatura real (°C)"); ax.set_ylabel("temperatura predicha (°C)")
raros = int((np.abs(p - y[PRUEBA]) > 20).sum())
ax.set_title(f"{peor}, entrenado con los datos sucios")
ax.text(0.03, 0.04, f"{raros} sitios de prueba con más de 20 °C de error", transform=ax.transAxes, color=ROJO, fontsize=11)
plt.show()

Menos de un 2 % de filas erróneas basta para estropear todos los modelos, y los más flexibles son los que más sufren. Un k-NN o un random forest **copian** la temperatura de los sitios parecidos: si uno de ellos dice −999 o 300, esa cifra se cuela en la predicción de todos sus vecinos (la figura de la derecha). La regresión lineal lo reparte entre todos los sitios y se degrada de forma más suave, pero tampoco sale gratis.

Los modelos potentes aprenden lo que hay en los datos, también los errores. Limpiar no es opcional.

---
# 4. Validación

El error de la sección 3 sale de **un único reparto** entre entrenamiento y prueba. ¿Cuánto se puede fiar uno de esa cifra?

### 4.1 Hold-out: un reparto, una cifra

Se repite el mismo experimento con muchos repartos distintos, todos con la misma fracción de prueba, y se mira cuánto cambia el error.

**Qué hacer:** elegir el algoritmo y ejecutar. El k-NN tarda unos segundos; el SVM, el XGBoost y la red, hasta un minuto.

In [ ]:
#@title 4.1 El mismo modelo con 25 repartos distintos
algoritmo_val = "k-NN (MAT)"  #@param ["Regresión lineal", "Ridge", "Árbol de decisión", "Random forest", "XGBoost", "k-NN (MAT)", "SVM", "Red neuronal"]
X, y = matriz("Todas las especies")
errores = []
for i in range(25):
    a, b = train_test_split(np.arange(len(y)), test_size=fraccion_prueba, random_state=1000 + i)
    errores.append(rmse(y[b], crear_modelo(algoritmo_val).fit(X[a], y[a]).predict(X[b])))
errores = np.array(errores)

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.scatter(errores, np.random.default_rng(1).uniform(-0.25, 0.25, len(errores)), s=70, color=AZUL, alpha=0.75, edgecolor="white")
ax.axvline(errores.mean(), color=TINTA, lw=1.5)
ax.axvspan(errores.min(), errores.max(), color=AZUL, alpha=0.07, lw=0)
ax.text(errores.mean(), 0.42, f"media {errores.mean():.2f} °C", ha="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none"))
ax.text(errores.min(), -0.47, f"{errores.min():.2f}", ha="center", color=AZUL); ax.text(errores.max(), -0.47, f"{errores.max():.2f}", ha="center", color=AZUL)
ax.set_ylim(-0.55, 0.55); ax.set_yticks([]); ax.grid(axis="y", visible=False)
ax.set_xlabel("error con datos nuevos (RMSE, °C)")
ax.set_title(f"{algoritmo_val}: cada punto es un reparto distinto de los mismos datos")
plt.show()

**El mismo modelo con los mismos datos** da errores distintos según qué sitios caigan en la prueba. Con un solo reparto, uno puede tener suerte (o mala suerte) sin saberlo. Si se comparan dos modelos y la diferencia es menor que esta franja, no se puede decir que uno sea mejor.

### 4.2 Validación cruzada (k-fold)

En vez de un reparto, **k repartos ordenados**: los datos se dividen en k partes, y cada parte hace una vez de prueba mientras las demás entrenan. Todos los sitios se usan para evaluar exactamente una vez, y salen k errores: una media y una dispersión.

**Qué hacer:** elegir k y el algoritmo y ejecutar.

In [ ]:
#@title 4.2 Validación cruzada
k = 5  #@param {type:"slider", min:3, max:10, step:1}
algoritmo_cv = "k-NN (MAT)"  #@param ["Regresión lineal", "Ridge", "Árbol de decisión", "Random forest", "XGBoost", "k-NN (MAT)", "SVM", "Red neuronal"]
X, y = matriz("Todas las especies")
cv = cross_validate(crear_modelo(algoritmo_cv), X, y, cv=KFold(k, shuffle=True, random_state=0),
                    scoring="neg_root_mean_squared_error", return_train_score=True)
eb, ea = -cv["test_score"], -cv["train_score"]
apuntar("4.2", algoritmo_cv, "Todas las especies", f"validación cruzada, k = {k}", ea.mean(), eb.mean(), np.nan)

fig, axs = plt.subplots(1, 2, figsize=(15, 0.55 * k + 1.6), gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.15})
ax = axs[0]
for i in range(k):
    for j in range(k):
        ax.add_patch(plt.Rectangle((j, k - 1 - i), 0.94, 0.8, color=ORO if i == j else AZUL, alpha=1 if i == j else 0.35, lw=0))
    ax.text(-0.15, k - 1 - i + 0.4, f"vuelta {i + 1}", ha="right", va="center")
ax.set_xlim(-1.3, k); ax.set_ylim(-0.75, k); ax.axis("off")
ax.text(0, -0.45, "■ entrena", color=AZUL, alpha=0.7, va="center"); ax.text(k * 0.3, -0.45, "■ prueba", color=ORO, va="center")
ax.set_title("Cómo se reparten los datos")
ax = axs[1]
ax.scatter(eb, np.arange(k)[::-1] + 0.4, s=80, color=ORO, zorder=3)
ax.axvline(eb.mean(), color=TINTA, lw=1.5)
ax.axvspan(eb.mean() - eb.std(), eb.mean() + eb.std(), color=ORO, alpha=0.12, lw=0)
ax.set_ylim(-0.75, k); ax.set_yticks([]); ax.grid(axis="y", visible=False)
ax.set_xlabel("error con datos nuevos (RMSE, °C)")
ax.set_title(f"Error de cada vuelta: {eb.mean():.2f} ± {eb.std():.2f} °C")
plt.show()

**Media y dispersión:** la forma honesta de dar el error de un modelo es «1,0 ± 0,05 °C», no «1,0 °C». Con pocas muestras, la validación cruzada es casi obligatoria: no se puede desperdiciar un 25 % de los datos en una sola prueba.

### 4.3 Validación por bloques: la trampa de la geografía

Los sitios cercanos se parecen: misma temperatura, asociaciones parecidas. Con repartos al azar, cada sitio de prueba tiene casi siempre un **vecino** en el entrenamiento, y el examen resulta más fácil de lo que será en la realidad.

La **validación por bloques** reparte **regiones enteras**: el mapa se divide en casillas de 30° × 30°, y cada casilla entera cae en entrenamiento o en prueba. Así se mide lo que se quiere saber: cómo funciona el modelo en un sitio **sin vecinos conocidos**, como un testigo de una cuenca poco muestreada.

**Qué hacer:** ejecutar la celda. Compara las dos validaciones con y sin latitud y longitud. Tarda unos 30 segundos.

In [ ]:
#@title 4.3 Al azar frente a por bloques
algoritmo_blq = "Random forest"  #@param ["Regresión lineal", "Ridge", "Árbol de decisión", "Random forest", "XGBoost", "k-NN (MAT)", "SVM", "Red neuronal"]
bloque = (np.floor((limpio["Latitude"] + 90) / 30) * 100 + np.floor((limpio["Longitude"] + 180) / 30)).astype(int).values
azar, bloques = KFold(5, shuffle=True, random_state=0), GroupKFold(5)
pliegue_azar, pliegue_blq = np.zeros(len(limpio), int), np.zeros(len(limpio), int)
for i, (_, b) in enumerate(azar.split(limpio)):
    pliegue_azar[b] = i
for i, (_, b) in enumerate(bloques.split(limpio, groups=bloque)):
    pliegue_blq[b] = i

res = {}
for p in ["Todas las especies", "Especies + latitud y longitud"]:
    X, y = matriz(p)
    for nombre, cv in (("al azar", azar), ("por bloques", bloques)):
        s = -cross_validate(crear_modelo(algoritmo_blq), X, y, cv=cv, groups=bloque, scoring="neg_root_mean_squared_error")["test_score"]
        res[(p, nombre)] = s
        apuntar("4.3", algoritmo_blq, p, f"validación {nombre}", np.nan, s.mean(), np.nan)

CINCO = ListedColormap([AZUL, ORO, ROJO, VERDE, TINTA])
fig = plt.figure(figsize=(15, 8))
for j, (pl, t) in enumerate(((pliegue_azar, "Al azar: los cinco grupos, mezclados"), (pliegue_blq, "Por bloques: regiones enteras"))):
    ax = fig.add_axes([0.02 + j * 0.49, 0.5, 0.47, 0.45])
    fondo_mapa(ax)
    ax.scatter(limpio["Longitude"], limpio["Latitude"], c=pl, cmap=CINCO, vmin=-0.5, vmax=4.5, s=5, lw=0)
    ax.set_title(t)
ax = fig.add_axes([0.25, 0.07, 0.5, 0.35])
pos = np.arange(2)
for j, (nombre, color) in enumerate((("al azar", AZUL), ("por bloques", ROJO))):
    m = [res[(p, nombre)].mean() for p in ["Todas las especies", "Especies + latitud y longitud"]]
    s = [res[(p, nombre)].std() for p in ["Todas las especies", "Especies + latitud y longitud"]]
    ax.bar(pos + (j - 0.5) * 0.38, m, yerr=s, width=0.36, color=color, label=f"validación {nombre}", capsize=4, error_kw={"ecolor": GRIS})
    for i, v in enumerate(m):
        ax.text(i + (j - 0.5) * 0.38, v + s[i] + 0.06, f"{v:.2f}", ha="center", va="bottom")
ax.set_xticks(pos); ax.set_xticklabels(["solo especies", "especies + latitud y longitud"]); ax.grid(axis="x", visible=False)
ax.set_ylim(0, max(v.mean() + v.std() for v in res.values()) * 1.35)
ax.set_ylabel("error con datos nuevos (RMSE, °C)"); ax.legend(loc="upper center", ncol=2)
ax.set_title(algoritmo_blq)
plt.show()

**Dos lecciones:**

1. **Por bloques, el error sube mucho.** Con repartos al azar, el modelo se apoyaba en vecinos casi idénticos. La cifra por bloques es la que se parece a lo que pasará al aplicarlo a un testigo nuevo.
2. **La ventaja de la latitud y la longitud se encoge por bloques.** Con repartos al azar parecían ayudar mucho; por bloques la mejora es mucho menor y queda dentro de la dispersión (las barras grises). Buena parte de esa «ayuda» era el modelo reconociendo el vecindario, no aprendiendo de los foraminíferos.

Elegir **cómo** se valida forma parte del diseño del modelo, y depende de para qué se vaya a usar. En la sesión 3 se lleva esto más lejos.

### 4.4 La tabla de todo lo probado

Cada modelo entrenado en el laboratorio (3.1), en la comparación de predictores (3.4) y en las validaciones (4.2 y 4.3) se ha ido apuntando aquí.

**Qué hacer:** ejecutar la celda. Se puede volver al laboratorio, probar más cosas y ejecutarla de nuevo.

In [ ]:
#@title 4.4 Resumen
tabla = pd.DataFrame(REGISTRO.values()).sort_values("error_datos_nuevos").reset_index(drop=True)
tabla.index += 1
tabla.columns = ["celda", "algoritmo", "predictores", "hiperparámetros / validación", "error entrenamiento (°C)",
                 "error datos nuevos (°C)", "R² datos nuevos"]
display(tabla.style.format(precision=2, na_rep="–")
        .bar(subset=["error datos nuevos (°C)"], color="#E6C27A", vmin=0)
        .set_properties(**{"font-size": "12px"}))

t = tabla.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 0.42 * len(t) + 1.2))
etiquetas = [f"{i}. {a} · {p} · {h}" for i, a, p, h in zip(t.index, t["algoritmo"], t["predictores"], t["hiperparámetros / validación"])]
ax.barh(etiquetas, t["error datos nuevos (°C)"], color=ORO, height=0.6, label="datos nuevos")
ax.scatter(t["error entrenamiento (°C)"], etiquetas, color=AZUL, s=40, zorder=3, label="entrenamiento")
ax.tick_params(axis="y", labelsize=9, length=0); ax.grid(axis="y", visible=False)
ax.set_xlabel("error (RMSE, °C)"); ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
ax.set_title("Los 15 mejores con datos nuevos", pad=30)
plt.show()

---
# Chuleta para llevarse

**El recorrido completo, en orden**

1. **Cargar y contar.** Filas y columnas, ¿son las que se esperaban?
2. **Dibujar antes de calcular.** Mapa, histograma, resumen estadístico. Un histograma ilegible es una alarma.
3. **Pasar controles sistemáticos.** Huecos, duplicados, rangos físicos, sumas, unidades, coordenadas y lo que se sepa del sistema.
4. **Corregir solo lo que tiene arreglo seguro.** Quitar el resto. Contar otra vez.
5. **Elegir predictores con criterio.** Presencia, abundancia, relación con el objetivo y redundancia. Cuidado con las variables que hacen trampa, como la geografía.
6. **Separar antes de entrenar.** Nunca evaluar con los mismos datos con los que se ha entrenado.
7. **Empezar por un modelo simple** (regresión lineal) como referencia, y solo complicar si mejora con datos nuevos.
8. **Ajustar los hiperparámetros mirando el error con datos nuevos**, nunca el de entrenamiento.
9. **Validar como se va a usar.** Validación cruzada siempre que se pueda; por bloques si el modelo se va a aplicar en sitios o épocas sin vecinos.
10. **Dar el error con su dispersión.** «1,0 ± 0,05 °C», no «1,0 °C».

**Errores típicos**

| Síntoma | Qué suele ser |
|---|---|
| R² de 0,99 o error casi nulo | Evaluado con los datos de entrenamiento, o una variable que «chiva» la respuesta |
| Mucho mejor en entrenamiento que en prueba | Sobreajuste: simplificar el modelo |
| Igual de malo en los dos | Infraajuste: modelo demasiado simple o predictores sin información |
| El error cambia mucho al repetir | Pocos datos o un solo reparto: usar validación cruzada |
| Bien al azar, mal por bloques | El modelo reconoce el vecindario, no la señal |

**Para usarlo con datos propios.** El cuaderno funciona con cualquier tabla con una columna objetivo y varias columnas predictoras. Lo que cambia son los controles de la sección 1, que dependen de lo que se sepa de cada sistema.

*Datos: ForCenS (Siccha y Kucera, 2017, Scientific Data 4, 170109) y World Ocean Atlas 2023 (NOAA). Mapa: Natural Earth.*